<a href="https://colab.research.google.com/github/hkrishnanrt/DSA-Activities/blob/main/Data_Acquisition_Case_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
population=pd.read_csv('/content/population.csv')
population

,country,population
0,United States,1278642419
1,China,792846414
2,India,1001406378
3,Japan,1206263687
4,Germany,428734972
5,Russia,420968276
6,Brazil,675094950
7,United Kingdom,674991378
8,France,434389014
9,Italy,254467210


In [ ]:
gdp=pd.read_excel('/content/gdp.xlsx')
gdp

,Country,GDP
0,United States,10450436373455
1,China,5086547998939
2,India,14540718118693
3,Japan,12473835857216
4,Germany,2038442396490
5,Russia,10987125474745
6,Brazil,8793052816619
7,United Kingdom,11799848684255
8,France,7758669752269
9,Italy,12907234326850


In [ ]:
internet_users=pd.read_json('/content/internet_users.json')
internet_users

,country,internet_users
0,United States,745595490
1,China,685275917
2,India,606850704
3,Japan,1112451555
4,Germany,246827121
5,Russia,162156967
6,Brazil,24027075
7,United Kingdom,266047041
8,France,216417853
9,Italy,87191493


In [ ]:
literacy_rate=pd.read_xml('/content/literacy_rate.xml')
literacy_rate

,name,literacy_rate
0,United States,72.7
1,China,82.2
2,India,80.3
3,Japan,97.5
4,Germany,92.9
5,Russia,89.1
6,Brazil,81.0
7,United Kingdom,82.9
8,France,97.6
9,Italy,83.7


In [ ]:
#Standardise country names across the dataset

population=population.rename(columns={'country':'Country'})
internet_users=internet_users.rename(columns={'country':'Country'})
literacy=literacy_rate.rename(columns={'name':'Country'})

In [ ]:
#Handle missing or erroneous data

for i, df in enumerate([population,gdp, internet_users,literacy], start=1):
    print(f"Dataset {i} has nulls? {df.isnull().values.any()}")


Dataset 1 has nulls? False
Dataset 2 has nulls? False
Dataset 3 has nulls? False
Dataset 4 has nulls? False


In [ ]:
#Create new columns such as internet penetration rate (internet users / population × 100)

internet_users = pd.merge(
    internet_users,
    population[["Country", "population"]],
    on="Country"
)

internet_users["internet_penetration_rate"] = (internet_users["internet_users"] / internet_users["population"]) * 100

internet_users=internet_users[["Country", "internet_users", "internet_penetration_rate"]]
internet_users

,Country,internet_users,internet_penetration_rate
0,United States,745595490,58.311493
1,China,685275917,86.432366
2,India,606850704,60.599844
3,Japan,1112451555,92.222917
4,Germany,246827121,57.571026
5,Russia,162156967,38.519997
6,Brazil,24027075,3.559066
7,United Kingdom,266047041,39.414880
8,France,216417853,49.821208
9,Italy,87191493,34.264333


In [ ]:
#Merge the datasets based on the country name

import pandas as pd

data = pd.merge(population, gdp, on="Country")

data = pd.merge(data, internet_users, on="Country")

data = pd.merge(data, literacy, on="Country")

data


,Country,population,GDP,internet_users,internet_penetration_rate,literacy_rate
0,United States,1278642419,10450436373455,745595490,58.311493,72.7
1,China,792846414,5086547998939,685275917,86.432366,82.2
2,India,1001406378,14540718118693,606850704,60.599844,80.3
3,Japan,1206263687,12473835857216,1112451555,92.222917,97.5
4,Germany,428734972,2038442396490,246827121,57.571026,92.9
5,Russia,420968276,10987125474745,162156967,38.519997,89.1
6,Brazil,675094950,8793052816619,24027075,3.559066,81.0
7,United Kingdom,674991378,11799848684255,266047041,39.414880,82.9
8,France,434389014,7758669752269,216417853,49.821208,97.6
9,Italy,254467210,12907234326850,87191493,34.264333,83.7


In [ ]:
#Handle unmatched or missing records


data.isnull().values.any()

np.False_

In [ ]:
validation_report = {
    "Missing values": data.isnull().sum().to_dict(),
    "Duplicate rows": data.duplicated().sum(),
    "Invalid populations": (data['population'] <= 0).sum(),
    "Invalid penetration rates": ((data['internet_penetration_rate'] < 0) |
                                  (data['internet_penetration_rate'] > 100)).sum(),
}
print(validation_report)


{'Missing values': {'Country': 0, 'population': 0, 'GDP': 0, 'internet_users': 0, 'internet_penetration_rate': 0, 'literacy_rate': 0}, 'Duplicate rows': np.int64(0), 'Invalid populations': np.int64(0), 'Invalid penetration rates': np.int64(0)}


In [ ]:
#Find countries with highest internet penetration rates

data.nlargest(5,'internet_penetration_rate')

,Country,population,GDP,internet_users,internet_penetration_rate,literacy_rate
3,Japan,1206263687,12473835857216,1112451555,92.222917,97.5
13,Spain,898664919,15275513099842,794943861,88.458317,60.6
1,China,792846414,5086547998939,685275917,86.432366,82.2
19,South Africa,916989541,3328747361982,739194223,80.610976,87.7
10,Canada,1438267572,8158351525190,894102645,62.165251,70.8


In [ ]:
#Calculate average literacy rate across countries

Average_literacy=data['literacy_rate'].mean()
Average_literacy.item()

78.43

In [ ]:
import sqlite3
conn=sqlite3.connect('students.db')
cursor=conn.cursor()

In [ ]:
# Load the dataframe into the database
data.to_sql('data', conn, if_exists='replace', index=False)

20

In [ ]:
cursor.execute("select * from data")

In [ ]:
cursor.fetchall()

[('United States',
  1278642419,
  10450436373455,
  745595490,
  58.311493418395656,
  72.7),
 ('China', 792846414, 5086547998939, 685275917, 86.4323663321759, 82.2),
 ('India', 1001406378, 14540718118693, 606850704, 60.59984411243684, 80.3),
 ('Japan', 1206263687, 12473835857216, 1112451555, 92.22291667974251, 97.5),
 ('Germany', 428734972, 2038442396490, 246827121, 57.57102571982394, 92.9),
 ('Russia', 420968276, 10987125474745, 162156967, 38.51999693202535, 89.1),
 ('Brazil', 675094950, 8793052816619, 24027075, 3.559066024712524, 81.0),
 ('United Kingdom',
  674991378,
  11799848684255,
  266047041,
  39.41487990384375,
  82.9),
 ('France', 434389014, 7758669752269, 216417853, 49.82120772511065, 97.6),
 ('Italy', 254467210, 12907234326850, 87191493, 34.26433331037032, 83.7),
 ('Canada', 1438267572, 8158351525190, 894102645, 62.165250917581005, 70.8),
 ('Australia', 439285667, 5444267975247, 201619113, 45.89703879412028, 71.6),
 ('South Korea',
  618608295,
  6562909541266,
  291770

In [ ]:
# Countries with highest internet penetration
cursor.execute("select Country,internet_penetration_rate from data order by internet_penetration_rate desc  limit 5  ")
cursor.fetchall()

[('Japan', 92.22291667974251),
 ('Spain', 88.45831679783196),
 ('China', 86.4323663321759),
 ('South Africa', 80.61097645605535),
 ('Canada', 62.165250917581005)]

In [ ]:
# Average literacy rate
cursor.execute("select avg(literacy_rate) from data")
cursor.fetchall()

[(78.43,)]

In [ ]:
#Correlation between literacy rate and internet penetration

correlation = data['literacy_rate'].corr(data['internet_penetration_rate'])
print("Correlation between literacy rate and internet penetration:", correlation)



Correlation between literacy rate and internet penetration: 0.23533009202707547


In [ ]:
#GDP per capita analysis
cursor.execute('SELECT Country,gdp,population,(gdp/population) as gdp_per_capita FROM data order BY gdp_per_capita DESC LIMIT 10;'
)
cursor.fetchall()

[('Indonesia', 12404144955318, 93409749, 132792),
 ('Italy', 12907234326850, 254467210, 50722),
 ('Russia', 10987125474745, 420968276, 26099),
 ('France', 7758669752269, 434389014, 17861),
 ('United Kingdom', 11799848684255, 674991378, 17481),
 ('Spain', 15275513099842, 898664919, 16998),
 ('India', 14540718118693, 1001406378, 14520),
 ('Brazil', 8793052816619, 675094950, 13024),
 ('Australia', 5444267975247, 439285667, 12393),
 ('South Korea', 6562909541266, 618608295, 10609)]

In [ ]:
#Which countries have a GDP per capita above $10,000?\
cursor.execute("select Country from data where (gdp/population)>10000")
cursor.fetchall()

[('India',),
 ('Japan',),
 ('Russia',),
 ('Brazil',),
 ('United Kingdom',),
 ('France',),
 ('Italy',),
 ('Australia',),
 ('South Korea',),
 ('Spain',),
 ('Indonesia',),
 ('Turkey',)]

In [ ]:
#What is the total population covered in the dataset?
cursor.execute('select sum(population)  from data')
cursor.fetchall()

[(14864685089,)]

In [ ]:
#Which countries have the lowest literacy rates, and how does that impact internet access?
cursor.execute("select Country,literacy_rate,internet_penetration_rate from data order by literacy_rate asc")
cursor.fetchall()

[('Saudi Arabia', 60.5, 11.36642948901554),
 ('Spain', 60.6, 88.45831679783196),
 ('South Korea', 66.4, 47.165660945429124),
 ('Argentina', 67.8, 29.415710388990107),
 ('Canada', 70.8, 62.165250917581005),
 ('Turkey', 71.4, 52.135862785790486),
 ('Australia', 71.6, 45.89703879412028),
 ('United States', 72.7, 58.311493418395656),
 ('Indonesia', 75.4, 4.9081622090644945),
 ('Mexico', 76.5, 20.74684477664874),
 ('India', 80.3, 60.59984411243684),
 ('Brazil', 81.0, 3.559066024712524),
 ('China', 82.2, 86.4323663321759),
 ('United Kingdom', 82.9, 39.41487990384375),
 ('Italy', 83.7, 34.26433331037032),
 ('South Africa', 87.7, 80.61097645605535),
 ('Russia', 89.1, 38.51999693202535),
 ('Germany', 92.9, 57.57102571982394),
 ('Japan', 97.5, 92.22291667974251),
 ('France', 97.6, 49.82120772511065)]

In [ ]:
#What are the top 5 wealthiest countries by total GDP, and how does that compare with population size?

cursor.execute("select Country,population,gdp from data order by gdp desc")
cursor.fetchall()

[('Spain', 898664919, 15275513099842),
 ('India', 1001406378, 14540718118693),
 ('Italy', 254467210, 12907234326850),
 ('Japan', 1206263687, 12473835857216),
 ('Indonesia', 93409749, 12404144955318),
 ('United Kingdom', 674991378, 11799848684255),
 ('Russia', 420968276, 10987125474745),
 ('United States', 1278642419, 10450436373455),
 ('Saudi Arabia', 958477463, 8855399647233),
 ('Brazil', 675094950, 8793052816619),
 ('Argentina', 1432830251, 8767336567320),
 ('Canada', 1438267572, 8158351525190),
 ('France', 434389014, 7758669752269),
 ('South Korea', 618608295, 6562909541266),
 ('Australia', 439285667, 5444267975247),
 ('China', 792846414, 5086547998939),
 ('South Africa', 916989541, 3328747361982),
 ('Mexico', 653061058, 2775281278756),
 ('Turkey', 247285876, 2581127745611),
 ('Germany', 428734972, 2038442396490)]

In [ ]:
#Find countries where internet users exceed 70% of the population.
cursor.execute("select Country from data where internet_users>(0.7*population) ")
cursor.fetchall()

[('China',), ('Japan',), ('Spain',), ('South Africa',)]

In [ ]:
#What is the average GDP per capita for countries with internet penetration above 50%?
cursor.execute("SELECT AVG(gdp * 1.0 / population) AS avg_gdp_per_capita FROM data WHERE (internet_users * 100.0 / population) > 50;")
cursor.fetchall()

[(8993.624949964475,)]

In [ ]:
#How many countries have a literacy rate above 90%, and what is their average internet penetration?
cursor.execute("SELECT COUNT(*) AS num_countries,AVG(internet_users * 100.0 / population) AS avg_internet_penetration FROM data WHERE literacy_rate > 90;")
cursor.fetchall()

[(3, 66.53838337489236)]